# Measure Discovery and Reconciliation
Inspect manual and inferred measures, precedence, group selection, controlled SQL, comparisons, approvals, and reports.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd
import yaml

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import TablePair, load_app_config
from dq_agent.context_store import read_context
from dq_agent.context_utils import (
    build_context_proposals, configure_workflow_logging, logged_step,
    workflow_paths, write_approval_workbook,
)
from dq_agent.measures import (
    compare_reconciliation_results, infer_measure_candidates, load_measure_configuration,
    measure_context_record, reconciliation_sql, resolve_measure_precedence,
    select_reconciliation_groups, validate_measure_metadata,
)
from dq_agent.query_engine import QueryGuard
from dq_agent.reporting import write_json, write_measure_reports

In [ ]:
RUN_ID = 'measure_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
USE_SAMPLE_DATA = True
EXECUTE_LIVE_QUERIES = False
config = load_app_config(ROOT)
paths = workflow_paths(config, RUN_ID, 'reconciliation')
logger = configure_workflow_logging(paths['log'], config.project.log_level)
guard = QueryGuard()
measure_config = load_measure_configuration(config)
sample = yaml.safe_load((ROOT / 'examples/measure_sample_metadata.yaml').read_text(encoding='utf-8'))
print('Run ID:', RUN_ID, 'Output:', paths['output'])

In [ ]:
with logged_step(logger, paths['checkpoint'], 'LOAD_MEASURE_CONFIGURATION', file=config.project.measures):
    configured = pd.DataFrame([item.model_dump() for item in measure_config.measures])
    logger.info('LOAD_MEASURE_CONFIGURATION measures=%s', len(configured))
display(configured)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'RETRIEVE_MEASURE_CONTEXT'):
    trusted = read_context(config, logger=logger)
    trusted_measures = trusted[trusted.context_type.isin(['measure', 'kpi'])] if not trusted.empty else pd.DataFrame()
    logger.info('RETRIEVE_MEASURE_CONTEXT records=%s', len(trusted_measures))
display(trusted_measures)

In [ ]:
pairs = [
    TablePair(pair_id='sample_fact_sales', mode='migration', source_catalog='main', source_schema='sales', source_table='fact_sales', target_project='your-gcp-project', target_dataset='analytics', target_table='fact_sales'),
    TablePair(pair_id='sample_fact_orders', mode='migration', source_catalog='main', source_schema='sales', source_table='fact_orders', target_project='your-gcp-project', target_dataset='analytics', target_table='fact_orders'),
]
mappings = {
    'sample_fact_sales': [
        {'source_column': 'sales_amount', 'target_column': 'sales_amount'},
        {'source_column': 'cost_amount', 'target_column': 'cost_amount'},
        {'source_column': 'sales_date', 'target_column': 'sales_date'},
        {'source_column': 'brand_id', 'target_column': 'brand_sid'},
        {'source_column': 'market_id', 'target_column': 'market_sid'},
        {'source_column': 'source_system', 'target_column': 'source_system'},
    ],
    'sample_fact_orders': [
        {'source_column': 'units_sold', 'target_column': 'units_sold'},
        {'source_column': 'profit_margin', 'target_column': 'profit_margin'},
        {'source_column': 'channel_id', 'target_column': 'channel_sid'},
    ],
}
metadata_by_pair = sample['tables']

In [ ]:
with logged_step(logger, paths['checkpoint'], 'INFER_MEASURES'):
    inferred = []
    for pair in pairs:
        metadata = metadata_by_pair[pair.pair_id]
        profiles = {
            key.split('|', 1)[1]: value for key, value in sample.get('profiles', {}).items()
            if key.startswith(pair.pair_id + '|')
        }
        rows = infer_measure_candidates(
            pair, metadata, mappings[pair.pair_id], profiles,
            measure_config.settings.max_inferred_per_table,
        )
        inferred.extend(rows)
        for row in rows:
            logger.info('INFER_MEASURES pair=%s measure=%s confidence=%s evidence=%s', pair.pair_id, row['measure_id'], row['confidence'], row['evidence']['score_components'])
display(pd.json_normalize(inferred, sep='.'))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'RESOLVE_MEASURE_PRECEDENCE'):
    resolved = []
    inferred_by_pair = {pair.pair_id: [row for row in inferred if row['pair_id'] == pair.pair_id] for pair in pairs}
    for pair in pairs:
        table_context = {'measures': []}
        selected = resolve_measure_precedence(pair, measure_config, table_context, inferred_by_pair[pair.pair_id])
        for measure in selected:
            measure.setdefault('pair_id', pair.pair_id)
            measure.setdefault('source_table', pair.source_name)
            measure.setdefault('target_table', pair.target_name)
            errors = validate_measure_metadata(measure, pair, metadata_by_pair[pair.pair_id])
            measure['metadata_errors'] = errors
            logger.info('RESOLVE_MEASURE_PRECEDENCE pair=%s measure=%s origin=%s confidence=%s errors=%s', pair.pair_id, measure['measure_id'], measure.get('origin'), measure.get('confidence'), errors)
            if not errors:
                resolved.append(measure)
display(pd.json_normalize(resolved, sep='.'))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'CREATE_APPROVAL_ITEM'):
    inferred_for_reuse = [row for row in resolved if row.get('origin') == 'inference']
    records = pd.DataFrame([
        measure_context_record(row, config, RUN_ID, config.project.measures, 'AGENT_INFERENCE')
        for row in inferred_for_reuse
    ])
    proposals = build_context_proposals(records, trusted, RUN_ID) if not records.empty else pd.DataFrame()
    approval_file = None
    if not proposals.empty:
        approval_file = paths['pending'] / f'measure_approvals_{RUN_ID}.xlsx'
        write_approval_workbook(approval_file, proposals)
        logger.info('CREATE_APPROVAL_ITEM records=%s output=%s', len(proposals), approval_file)
print('Approval file:', approval_file)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'SELECT_RECONCILIATION_GROUPS'):
    generated = []
    groupings_by_measure = {}
    pair_by_id = {pair.pair_id: pair for pair in pairs}
    for measure in resolved:
        pair = pair_by_id[measure['pair_id']]
        groups = select_reconciliation_groups(
            measure, metadata_by_pair[pair.pair_id], mappings[pair.pair_id],
            measure_config.settings.max_groupings_per_measure,
        )
        groupings_by_measure[(pair.pair_id, measure['measure_id'])] = groups
        for grouping in groups:
            for side in ('source', 'target'):
                sql = reconciliation_sql(measure, pair, side, grouping, [], measure_config.settings.maximum_group_cardinality)
                allowed = {pair.source_name} if side == 'source' else {pair.target_name}
                checked = guard.validate(sql, 'databricks' if side == 'source' else 'bigquery', allowed)
                generated.append({'pair_id': pair.pair_id, 'measure_id': measure['measure_id'], 'grouping_id': grouping['grouping_id'], 'side': side, 'sql': checked})
                logger.info('GENERATE_RECONCILIATION_QUERY pair=%s measure=%s grouping=%s side=%s', pair.pair_id, measure['measure_id'], grouping['grouping_id'], side)
generated_sql = pd.DataFrame(generated)
display(generated_sql)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'EXECUTE_RECONCILIATION'):
    results = []
    available = sample.get('reconciliation_results', {})
    for measure in resolved:
        pair = pair_by_id[measure['pair_id']]
        for grouping in groupings_by_measure[(pair.pair_id, measure['measure_id'])]:
            prefix = f"{pair.pair_id}|{measure['measure_id']}|{grouping['grouping_id']}"
            if prefix + '|source' not in available or prefix + '|target' not in available:
                continue
            source = pd.DataFrame(available[prefix + '|source'])
            target = pd.DataFrame(available[prefix + '|target'])
            compared = compare_reconciliation_results(measure, grouping, source, target)
            for row in compared:
                row.update({'pair_id': pair.pair_id, 'rule_id': f"{pair.pair_id}__measure__{measure['measure_id']}__{grouping['grouping_id']}", 'type': 'measure_reconciliation', 'category': 'measure_reconciliation', 'origin': measure.get('origin')})
                results.append(row)
                logger.info('EXECUTE_RECONCILIATION pair=%s measure=%s grouping=%s status=%s difference=%s', pair.pair_id, measure['measure_id'], grouping['grouping_id'], row['status'], row['absolute_difference'])
reconciliation_results = pd.DataFrame(results)
display(reconciliation_results)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'WRITE_RECONCILIATION_RESULTS'):
    write_json(paths['logs'] / 'resolved_measures.json', resolved)
    write_json(paths['logs'] / 'measure_groupings.json', {f'{key[0]}|{key[1]}': value for key, value in groupings_by_measure.items()})
    write_json(paths['logs'] / 'measure_reconciliation.json', results)
    report_paths = write_measure_reports(paths['output'], resolved, results)
    logger.info('WRITE_RECONCILIATION_RESULTS measures=%s results=%s outputs=%s', len(resolved), len(results), report_paths)
report_paths

The sample demonstrates a manual measure overriding inference, controlled expressions, mapped grouping names, guarded SQL, reconciliation, and approval. Run `07_rca_and_context_learning.ipynb` next.